# Feature Engineering — Site Suitability Recommender

Normalize criteria to 0-1 scale, compute proximity metrics, and extract
terrain features for MCDA scoring.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial.distance import cdist
from sklearn.preprocessing import MinMaxScaler

## Feature Engineering Steps

1. Compute distance-to-water for each candidate site
2. Compute distance-to-road for accessibility scoring
3. Extract elevation and slope at each site
4. Normalize all criteria to 0-1 scale
5. Assemble suitability criteria matrix

In [ ]:
# Load data
sites = gpd.read_file('../data/raw/candidate_sites.shp')
water = gpd.read_file('../data/raw/water_bodies.shp')
roads = gpd.read_file('../data/raw/road_network.shp')

# Proximity to water
sites['dist_water'] = sites.geometry.apply(
    lambda pt: water.geometry.distance(pt).min()
)

# Proximity to roads
sites['dist_road'] = sites.geometry.apply(
    lambda pt: roads.geometry.distance(pt).min()
)

print(f'Distance to water: {sites.dist_water.describe()}')
print(f'Distance to road: {sites.dist_road.describe()}')

In [ ]:
# Normalize criteria to 0-1
criteria_cols = ['soil_quality', 'elevation', 'slope', 'ndvi', 'dist_water', 'dist_road']
scaler = MinMaxScaler()
sites[criteria_cols] = scaler.fit_transform(sites[criteria_cols])

# Invert distance features (closer = better)
sites['water_proximity'] = 1 - sites['dist_water']
sites['road_proximity'] = 1 - sites['dist_road']

sites.to_file('../data/processed/sites_normalized.shp')
print(f'Normalized criteria matrix: {sites[criteria_cols].shape}')
sites[criteria_cols].describe()